In [4]:
import pandas as pd
import tkinter as tk
from tkinter import messagebox
import os

# === Load your local word count CSV ===
csv_path = r"/graph/jupyter/counting\word_counts.csv"
df = pd.read_csv(csv_path)

df = df[df["count"] > 5].reset_index()
df["index"] = df["index"] + 1  # make index 1-based

if df.empty:
    print("No stopwords found")
    exit()
else:
    print(f"Found {len(df)} stopwords to classify.")

# === Output CSV for classified stopwords ===
output_path = r"E:\Github\uit_chatbot\graph\jupyter\stopwords.csv"

# Create file if not exists
if not os.path.exists(output_path):
    pd.DataFrame(columns=["index", "word"]).to_csv(output_path, index=False, encoding="utf-8-sig")

# === Load already classified stopwords ===
classified = set()
try:
    existing_df = pd.read_csv(output_path)
    classified = set(existing_df["word"].astype(str).tolist())
except Exception as e:
    print("Warning: could not load existing stopwords.csv:", e)

# === Filter out already classified words ===
remaining_df = df[~df["word"].isin(classified)].reset_index(drop=True)

if remaining_df.empty:
    print("All stopwords already classified.")
    exit()
else:
    print(f"{len(remaining_df)} stopwords remaining to classify.")

# === GUI setup ===
current_index = 0

def update_word():
    """Update the displayed word."""
    if current_index < len(remaining_df):
        idx = remaining_df.loc[current_index, "index"]
        word = remaining_df.loc[current_index, "word"]
        word_label.config(text=f"{idx}. {word}")
    else:
        messagebox.showinfo("Done", "All stopwords have been classified!")
        root.destroy()

def classify_stopword(event=None):
    """Save current word (with original index) to stopword.csv and go next."""
    global current_index
    if current_index >= len(remaining_df):
        return
    idx = remaining_df.loc[current_index, "index"]
    word = remaining_df.loc[current_index, "word"]
    new_entry = pd.DataFrame([[idx, word]], columns=["index", "word"])
    new_entry.to_csv(output_path, mode='a', header=False, index=False, encoding="utf-8-sig")
    current_index += 1
    update_word()

def skip_word(event=None):
    """Skip current word."""
    global current_index
    if current_index >= len(remaining_df):
        return
    current_index += 1
    update_word()

# === Build Tkinter window ===
root = tk.Tk()
root.title("Vietnamese Stopword Classifier")
root.geometry("400x200")

word_label = tk.Label(root, text="", font=("Arial", 20))
word_label.pack(pady=20)

btn_frame = tk.Frame(root)
btn_frame.pack(pady=20)

btn_stop = tk.Button(btn_frame, text="Stopword (Enter)", width=16, command=classify_stopword, bg="lightgreen")
btn_stop.pack(side="left", padx=10)

btn_skip = tk.Button(btn_frame, text="Skip (Backspace)", width=16, command=skip_word, bg="lightgray")
btn_skip.pack(side="right", padx=10)

# === Keyboard shortcuts ===
root.bind("<Return>", classify_stopword)     # Enter key → Stopword
root.bind("<BackSpace>", skip_word)          # Backspace key → Skip

update_word()
root.mainloop()

Found 755 stopwords to classify.

755 stopwords remaining to classify.
